In [ ]:
# Задание 4. SQL-запросы на основе описанных таблиц в ClickHouse

**Схемы данных:**
*   `dwh.products` — информация о товаре (`itemid`, `name_ax`, `brand_name_ax`, `category`)
*   `dwh.sales` — данные о продажах (`sale_id`, `date`, `itemid`, `qty`, `price`)
*   `dwh.stock` — данные об остатках (`date`, `itemid`, `qty`)

**Важное условие:**
*   В итоговом результате должны быть только товары, у которых были продажи за последний месяц.
*   Прошлый месяц должен вычисляться динамически в коде, а не хардкодиться (например, не `'2025-05-01'`).

In [ ]:
WITH 
    -- Динамическое определение последнего полного месяца
    toStartOfMonth(now() - INTERVAL 1 MONTH) AS last_month_start,
    toStartOfMonth(now()) AS current_month_start,
    last_month_start - INTERVAL 2 MONTH AS three_months_ago_start
SELECT 
    p.itemid AS "Код товара",
    p.name_ax AS "Название товара",
    p.brand_name_ax AS "Бренд",
    p.category AS "Категория",
    -- 5. Сумма продаж в шт. за последний месяц
    sumIf(s.qty, s.date >= last_month_start AND s.date < current_month_start) AS "Продажи шт. (мес)",
    
    -- 6. Сумма продаж в рублях за последний месяц
    sumIf(s.qty * s.price, s.date >= last_month_start AND s.date < current_month_start) AS "Продажи руб. (мес)",

    -- 7. Средняя цена за последний месяц
    avgIf(s.price, s.date >= last_month_start AND s.date < current_month_start) AS "Средняя цена (мес)",
    
    -- 8. Средний объем продаж в шт. за последние 3 месяца
    -- (Суммируем продажи за 3 месяца и делим на 3)
    sumIf(s.qty, s.date >= three_months_ago_start AND s.date < current_month_start) / 3 AS "Средний объем (3 мес)",
    
    -- 9. Разница между текущим остатком и средним объемом
    -- Берем последний остаток из таблицы stock
    anyLast(st.qty) - (sumIf(s.qty, s.date >= three_months_ago_start AND s.date < current_month_start) / 3) AS "Разница (Остаток - Средний объем)"

FROM dwh.products AS p
-- Присоединяем продажи за последние 3 месяца
LEFT JOIN dwh.sales AS s ON p.itemid = s.itemid 
    AND s.date >= three_months_ago_start 
    AND s.date < current_month_start
-- Присоединяем остатки (предполагаем, что в stock лежат актуальные остатки на текущую дату)
LEFT JOIN dwh.stock AS st ON p.itemid = st.itemid
GROUP BY 
    p.itemid, p.name_ax, p.brand_name_ax, p.category
HAVING 
    -- Фильтр: были продажи за последний месяц
    sumIf(s.qty, s.date >= last_month_start AND s.date < current_month_start) > 0

In [ ]:
2) Вывести топ 5 брендов

**Требуется:**
*   Вывести топ 5 брендов по объему продаж в рублях за последний месяц.
*   Сравнить с объемом продаж за предыдущий месяц.

In [ ]:
WITH 
    toStartOfMonth(now() - INTERVAL 1 MONTH) AS last_month_start,
    toStartOfMonth(now()) AS current_month_start,
    last_month_start - INTERVAL 1 MONTH AS prev_month_start
SELECT 
    p.brand_name_ax AS "Бренд",
    -- Продажи за последний месяц
    sumIf(s.qty * s.price, s.date >= last_month_start AND s.date < current_month_start) AS "Выручка (последний месяц)",
    
    -- Продажи за предыдущий месяц
    sumIf(s.qty * s.price, s.date >= prev_month_start AND s.date < last_month_start) AS "Выручка (предыдущий месяц)",
    -- Разница (динамика)
    "Выручка (последний месяц)" - "Выручка (предыдущий месяц)" AS "Динамика (руб)"
FROM dwh.sales AS s
JOIN dwh.products AS p ON s.itemid = p.itemid
WHERE 
    s.date >= prev_month_start AND s.date < current_month_start
GROUP BY 
    p.brand_name_ax
ORDER BY 
    "Выручка (последний месяц)" DESC
LIMIT 5;